In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")


In [ ]:
from pathlib import Path
import os


def resolve_data_path(filename: str) -> str:
    """
    Resolve data files in local Jupyter or Google Colab.
    In Colab, mount Google Drive first and place data under:
    /content/drive/MyDrive/feature-selection-experiments/data
    """
    candidates = []

    env_data_dir = os.environ.get("FEATURE_SELECTION_DATA_DIR")
    if env_data_dir:
        candidates.append(Path(env_data_dir) / filename)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
        Path("/content/data") / filename,
        Path("/content/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/data") / filename,
        Path(r"D:\feature-selection-experiments\data") / filename,
    ])

    for path in candidates:
        if path.exists():
            return str(path)

    search_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.glob(f"**/{filename}"))
        for match in matches:
            if match.parent.name == "data":
                return str(match)
        if matches:
            return str(matches[0])

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n{checked}\n\n"
        "Upload the data folder to /content/data, or mount Google Drive and place it at "
        "/content/drive/MyDrive/feature-selection-experiments/data."
    )


DATA_DIR = Path(resolve_data_path("X_train.csv")).parent
if Path("/content/drive/MyDrive").exists():
    RESULT_DIR = Path("/content/drive/MyDrive/feature-selection-experiments/results/pso-dt")
else:
    RESULT_DIR = Path.cwd() / "results" / "pso-dt"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = RESULT_DIR


def output_path(filename: str) -> str:
    return str(RESULT_DIR / filename)


print(f"Using DATA_DIR: {DATA_DIR}")
print(f"Using RESULT_DIR: {RESULT_DIR}")


In [ ]:
import importlib
import subprocess
import sys


def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name)
    except Exception:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            pip_name or import_name,
        ])


# Do not uninstall/re-pin numpy or pandas in Colab; that can break scipy/sklearn.
ensure_package("mealpy")

try:
    import numpy
    import pandas
    import scipy
    import sklearn
    print("Dependency check OK")
    print("numpy:", numpy.__version__)
    print("pandas:", pandas.__version__)
    print("scipy:", scipy.__version__)
    print("sklearn:", sklearn.__version__)
except Exception as exc:
    print("sklearn/scipy stack looks broken:", repr(exc))
    print("Repairing numpy/pandas/scipy/scikit-learn stack. Restart runtime after this cell if import still fails.")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "joblib",
        "threadpoolctl",
        "mealpy",
    ])


In [3]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import time
from mealpy import FloatVar, PSO
from sklearn.tree import DecisionTreeClassifier
import torch.nn as nn
import torch.nn.functional as F
import csv
import json
from typing import Tuple, List, Dict, Optional

In [4]:
def load_data(
    x_path: str,
    y_path: str
) -> Tuple[pd.DataFrame, np.ndarray]:
    X = pd.read_csv(x_path)
    y_df = pd.read_csv(y_path)

    if y_df.shape[1] != 1:
        raise ValueError("y_train.csv phải chỉ có đúng 1 cột nhãn.")

    y = y_df.iloc[:, 0].values

    # Encode nhãn nếu là chuỗi
    if y.dtype == object or str(y.dtype).startswith("string"):
        le = LabelEncoder()
        y = le.fit_transform(y)

    return X, y

In [8]:
X_path = resolve_data_path("X_train.csv")
y_path = resolve_data_path("y_train.csv")

X, y = load_data(X_path, y_path)
X_train_fs, X_test_fs, y_train_fs, y_test_fs = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:


def hamming_distance_ratio(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Return the normalized Hamming distance between two binary masks."""
    mask_a = np.asarray(mask_a).astype(int)
    mask_b = np.asarray(mask_b).astype(int)
    if mask_a.shape != mask_b.shape:
        raise ValueError("Two masks must have the same shape.")
    return float(np.mean(mask_a != mask_b))


def continuous_population_to_masks(
    population: np.ndarray,
    threshold: float = 0.5,
    min_features: int = 1,
    use_sigmoid: bool = False
) -> np.ndarray:
    """Convert a continuous population to binary feature masks for metric tracking."""
    values = 1.0 / (1.0 + np.exp(-population)) if use_sigmoid else np.asarray(population)
    masks = (values > threshold).astype(int)

    if min_features > 0:
        for i in range(masks.shape[0]):
            if masks[i].sum() < min_features:
                top_idx = np.argsort(values[i])[-min_features:]
                masks[i, top_idx] = 1

    return masks


def compute_population_diversity(population_masks: np.ndarray) -> float:
    """Average pairwise Hamming distance among all masks in the current population."""
    population_masks = np.asarray(population_masks).astype(int)
    n_individuals = population_masks.shape[0]
    if n_individuals < 2:
        return 0.0

    total_distance = 0.0
    pair_count = 0
    for i in range(n_individuals):
        for j in range(i + 1, n_individuals):
            total_distance += hamming_distance_ratio(population_masks[i], population_masks[j])
            pair_count += 1

    return float(total_distance / pair_count)


def compute_exploration_exploitation(
    new_masks: np.ndarray,
    previous_masks: np.ndarray,
    similarity_threshold: float = 0.1
) -> Tuple[float, float]:
    """
    Exploration is the fraction of new masks whose nearest previous neighbor
    is farther than the threshold. Exploitation is the complementary fraction.
    """
    new_masks = np.asarray(new_masks).astype(int)
    previous_masks = np.asarray(previous_masks).astype(int)
    if len(new_masks) == 0 or len(previous_masks) == 0:
        return 0.0, 0.0

    exploration_count = 0
    for new_mask in new_masks:
        nearest_distance = min(
            hamming_distance_ratio(new_mask, previous_mask)
            for previous_mask in previous_masks
        )
        if nearest_distance > similarity_threshold:
            exploration_count += 1

    exploration = exploration_count / len(new_masks)
    exploitation = 1.0 - exploration
    return float(exploration), float(exploitation)


def extract_solution_vector(individual):
    """Extract a solution vector from common mealpy Agent/history representations."""
    if hasattr(individual, "solution"):
        return np.asarray(individual.solution, dtype=float)
    if isinstance(individual, dict):
        for key in ("solution", "position", "pos"):
            if key in individual:
                return np.asarray(individual[key], dtype=float)
    if isinstance(individual, (list, tuple)) and len(individual) > 0:
        first = individual[0]
        if np.asarray(first).ndim > 0:
            return np.asarray(first, dtype=float)
    return np.asarray(individual, dtype=float)


def extract_population_matrix(population):
    """Convert a mealpy history population entry to a 2D numpy matrix."""
    return np.asarray([extract_solution_vector(ind) for ind in population], dtype=float)


def align_history(values, n: int, fill_value: float = 0.0) -> List[float]:
    values = list(values)
    if len(values) >= n:
        return values[:n]
    return values + [fill_value] * (n - len(values))


def save_pso_history(model):
    n_epochs = len(model.history.list_global_best_fit)
    diversity_mealpy = align_history(model.history.list_diversity, n_epochs)
    exploration_mealpy = align_history(model.history.list_exploration, n_epochs)
    exploitation_mealpy = align_history(model.history.list_exploitation, n_epochs)

    history_data = {
        "list_global_best": model.history.list_global_best,
        "list_current_best": model.history.list_current_best,
        "list_global_worst": model.history.list_global_worst,
        "list_current_worst": model.history.list_current_worst,
        "list_epoch_time": model.history.list_epoch_time,
        "list_global_best_fit": model.history.list_global_best_fit,
        "list_current_best_fit": model.history.list_current_best_fit,
        "list_diversity": model.history.list_diversity,
        "list_exploitation": model.history.list_exploitation,
        "list_exploration": model.history.list_exploration,
        "list_population": model.history.list_population,
    }

    history_df = pd.DataFrame({
        "epoch": list(range(1, len(model.history.list_global_best_fit) + 1)),
        "global_best_fit": model.history.list_global_best_fit,
        "current_best_fit": model.history.list_current_best_fit,
        "diversity": diversity_mealpy,
        "exploration": exploration_mealpy,
        "exploitation": exploitation_mealpy,
        "diversity_source": ["mealpy"] * n_epochs,
        "exploration_source": ["mealpy"] * n_epochs,
        "exploitation_source": ["mealpy"] * n_epochs,
        "epoch_time": align_history(model.history.list_epoch_time, n_epochs)
    })

    history_df.to_csv(output_path("optimization_metrics_pso_with_formulas.csv"), index=False)
    print("[Metrics] Saved optimization_metrics_pso_with_formulas.csv")

In [21]:
a = 0.5
num_features = X.shape[1]

def fitness_function(solution):
    selected_features = np.where(solution > 0.5)[0]

    if len(selected_features) == 0:
        return 1.0

    X_train_fs_selected = X_train_fs.iloc[:, selected_features]
    X_test_fs_selected = X_test_fs.iloc[:, selected_features]

    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train_fs_selected, y_train_fs)
    predictions = model.predict(X_test_fs_selected)
    accuracy = accuracy_score(y_test_fs, predictions)

    cp = accuracy
    sf = len(selected_features)
    tf = num_features

    fitness_value = a * (1 - cp) + (1 - a) * (1 - sf / tf)

    return fitness_value

In [22]:
problem_dict = {
    "bounds": FloatVar(lb=(0.,) * num_features, ub=(1.,) * num_features, name="delta"),
    "obj_func": fitness_function,
    "minmax": "min",
}

pso_model = PSO.OriginalPSO(epoch=50, pop_size=20, c1=1.5, c2=1.5, w=0.7, verbose = True)
start_train = time.time()
g_best = pso_model.solve(problem_dict)
end_train = time.time()
print(f"Feature selection time: {end_train - start_train:.4f} seconds")

2026/04/11 05:33:07 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: OriginalPSO(epoch=50, pop_size=20, c1=1.5, c2=1.5, w=0.7)
2026/04/11 05:36:08 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 1, Current best: 0.443875, Global best: 0.443875, Runtime: 91.71091 seconds
2026/04/11 05:37:38 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 2, Current best: 0.4438125, Global best: 0.4438125, Runtime: 89.97366 seconds
2026/04/11 05:39:03 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 3, Current best: 0.4438125, Global best: 0.4438125, Runtime: 85.27830 seconds
2026/04/11 05:40:29 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 4, Current best: 0.44334375, Global best: 0.44334375, Runtime: 86.01891 seconds
2026/04/11 05:41:56 AM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 5, Current best: 0.440625, Global best: 0.440625, Runtime: 87.30717 seconds
2026/04/11 05:43:24 AM, INFO, mealpy.swarm_based

Feature selection time: 4702.0603 seconds


In [28]:
save_pso_history(pso_model)
selected_features = np.where(g_best.solution > 0.5)[0]

with open(output_path('selected_indices.csv'), 'w') as f:
    for idx in selected_features:
        f.write(f"{idx}\n")

print(f"Selected {len(selected_features)} features out of {num_features}")

Selected 557 features out of 1000


In [29]:
import pandas as pd

selected_features = pd.read_csv(output_path("selected_indices.csv"), header=None)[0].values

X_train = pd.read_csv(resolve_data_path("X_train.csv"))
X_val = pd.read_csv(resolve_data_path("X_val.csv"))
X_test = pd.read_csv(resolve_data_path("X_test.csv"))

y_train = pd.read_csv(resolve_data_path("y_train.csv")).iloc[:, 0].values
y_val = pd.read_csv(resolve_data_path("y_val.csv")).iloc[:, 0].values
y_test = pd.read_csv(resolve_data_path("y_test.csv")).iloc[:, 0].values

X_train_selected = X_train.iloc[:, selected_features].copy()
X_val_selected = X_val.iloc[:, selected_features].copy()
X_test_selected = X_test.iloc[:, selected_features].copy()

print("Train selected shape:", X_train_selected.shape)
print("Val selected shape:", X_val_selected.shape)
print("Test selected shape:", X_test_selected.shape)

Train selected shape: (16000, 557)
Val selected shape: (2000, 557)
Test selected shape: (2000, 557)


In [30]:
def compute_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {"model": model_name}
    for cls in CLASS_NAMES:
        for m in ["precision","recall","f1-score"]:
            col = f"{cls}_{m.replace('-score','')}"
            row[col] = round(report[cls][m], 4)
    for avg in ["macro avg","weighted avg"]:
        prefix = avg.replace(" avg","")
        for m in ["precision","recall","f1-score"]:
            col = f"{prefix}_{m.replace('-score','')}"
            row[col] = round(report[avg][m], 4)
    row["accuracy"] = round(report["accuracy"], 4)
    return row

In [31]:
all_results = [] 

from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.naive_bayes       import GaussianNB
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import LinearSVC, SVC
from sklearn.model_selection   import GridSearchCV
from sklearn.pipeline          import Pipeline
import numpy as np

unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

print("N_CLASSES :", N_CLASSES)
print("CLASS_NAMES:", CLASS_NAMES)

ML_CONFIGS = [
    (
        "KNN",
        KNeighborsClassifier(),
        {"n_neighbors":[3,5,7,11], "weights":["uniform","distance"],
         "metric":["euclidean","manhattan"]}
    ),
    (
        "DecisionTree",
        DecisionTreeClassifier(random_state=42),
        {"max_depth":[None,10,20,30], "min_samples_split":[2,5,10],
         "criterion":["gini","entropy"]}
    ),
    (
        "RandomForest",
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,200], "max_depth":[None,10,20],
         "min_samples_split":[2,5]}
    ),
    (
        "GaussianNB",
        GaussianNB(),
        {"var_smoothing":[1e-9,1e-8,1e-7,1e-6]}
    ),
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        {"C":[0.01,0.1,1,10], "solver":["lbfgs","saga"],
         "multi_class":["multinomial"]}
    ),
    (
        "LinearSVM",
        LinearSVC(max_iter=2000, random_state=42),
        {"C":[0.01,0.1,1,10]}
    ),
    (
        "RBF_SVM",
        SVC(kernel="rbf", random_state=42),
        {"C":[0.1,1,10], "gamma":["scale","auto"]}
    ),
]

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

all_results = []
report_rows = []

for name, estimator, param_grid in ML_CONFIGS:
    print(f"\n[{name}] GridSearch đang chạy...")

    gs = GridSearchCV(
        estimator,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )

    gs.fit(X_train_selected, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test_selected)

    print(f"  Best params : {gs.best_params_}")
    print(f"  CV F1-macro : {gs.best_score_:.4f}")

    # report dạng dict thay vì in text thô
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True
    )

    # Gom toàn bộ report của model thành 1 hàng
    flat_row = {"model": name}

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):  # các class, macro avg, weighted avg
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:  # accuracy là scalar
            flat_row[label] = metrics

    flat_row["best_params"] = str(gs.best_params_)
    flat_row["cv_f1_macro"] = gs.best_score_
    report_rows.append(flat_row)

    # phần cũ của bạn nếu vẫn muốn giữ
    row = compute_metrics(y_test, y_pred, name)
    row["best_params"] = str(gs.best_params_)
    all_results.append(row)

# Bảng classification report: mỗi hàng là 1 model
classification_report_df = pd.DataFrame(report_rows)

# Sắp xếp cột cho dễ nhìn
first_cols = ["model", "accuracy", "cv_f1_macro", "best_params"]
other_cols = [c for c in classification_report_df.columns if c not in first_cols]
classification_report_df = classification_report_df[first_cols + other_cols]

display(classification_report_df)

pd.DataFrame(all_results).to_csv(
    output_path("all_results_models.csv"),
    index=False,
    encoding="utf-8-sig"
)

N_CLASSES : 8
CLASS_NAMES: ['0', '1', '2', '3', '4', '5', '6', '7']

[KNN] GridSearch đang chạy...
  Best params : {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
  CV F1-macro : 0.5540

[DecisionTree] GridSearch đang chạy...
  Best params : {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 2}
  CV F1-macro : 0.6348

[RandomForest] GridSearch đang chạy...
  Best params : {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
  CV F1-macro : 0.7640

[GaussianNB] GridSearch đang chạy...
  Best params : {'var_smoothing': 1e-06}
  CV F1-macro : 0.3576

[LogisticRegression] GridSearch đang chạy...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

  Best params : {'C': 0.01, 'multi_class': 'multinomial', 'solver': 'lbfgs'}
  CV F1-macro : 0.7693

[LinearSVM] GridSearch đang chạy...
  Best params : {'C': 0.01}
  CV F1-macro : 0.7692

[RBF_SVM] GridSearch đang chạy...
  Best params : {'C': 10, 'gamma': 'auto'}
  CV F1-macro : 0.8371


,model,accuracy,cv_f1_macro,best_params,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,...,7_f1-score,7_support,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support
0,KNN,0.5815,0.553973,"{'metric': 'manhattan', 'n_neighbors': 11, 'we...",0.565657,0.224,0.320917,250.0,0.408240,0.872,...,0.680556,250.0,0.624271,0.5815,0.575455,2000.0,0.624271,0.5815,0.575455,2000.0
1,DecisionTree,0.6410,0.634753,"{'criterion': 'entropy', 'max_depth': 10, 'min...",0.538462,0.448,0.489083,250.0,0.630225,0.784,...,0.617587,250.0,0.639419,0.6410,0.638223,2000.0,0.639419,0.6410,0.638223,2000.0
2,RandomForest,0.7780,0.763979,"{'max_depth': None, 'min_samples_split': 2, 'n...",0.739130,0.544,0.626728,250.0,0.773852,0.876,...,0.807692,250.0,0.782998,0.7780,0.775827,2000.0,0.782998,0.7780,0.775827,2000.0
3,GaussianNB,0.3780,0.357598,{'var_smoothing': 1e-06},0.300000,0.108,0.158824,250.0,0.346709,0.864,...,0.506460,250.0,0.432318,0.3780,0.364150,2000.0,0.432318,0.3780,0.364150,2000.0
4,LogisticRegression,0.7815,0.769286,"{'C': 0.01, 'multi_class': 'multinomial', 'sol...",0.687783,0.608,0.645435,250.0,0.828244,0.868,...,0.806723,250.0,0.780738,0.7815,0.780327,2000.0,0.780738,0.7815,0.780327,2000.0
5,LinearSVM,0.7755,0.769181,{'C': 0.01},0.685864,0.524,0.594104,250.0,0.808118,0.876,...,0.803313,250.0,0.772901,0.7755,0.772252,2000.0,0.772901,0.7755,0.772252,2000.0
6,RBF_SVM,0.8410,0.837107,"{'C': 10, 'gamma': 'auto'}",0.745763,0.704,0.724280,250.0,0.854902,0.872,...,0.856557,250.0,0.841587,0.8410,0.840718,2000.0,0.841587,0.8410,0.840718,2000.0
